# Tutorial on using xarray DataArray objects as arguments for xclim Indicator objects

Reading the xclim documentation I found that xclim Indicator objects can accept `xarray.DataArray` objects as arguments for some keyword parameters, when the indicator is called for computation. What is the implication of this functionality on the evaluation of a specific indicator? Does this parallelise the computation? If so, how?

I take indicator [heat_wave_total_length](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_total_length) as example. Its documentation does not explain clearly how it works, but [the changelog](https://xclim.readthedocs.io/en/stable/changes.html#id52) of the implementation contains some additional details.

In this notebook I experiment with this functionality, to understand if it may be useful for my goal: to evaluate in parallel an indicator over all combinations of values for the input parameters, if they are stored as coordinates in the input xarray Dataset object.

## Set up

In [ ]:
import numpy  # Version 1.26.4.
import xarray  # Version 2024.5.0.
import xclim  # Version 0.49.0.

import utilities

In [ ]:
ds_hourly = utilities.open_dataset()
indicator = xclim.core.indicator.registry["HEAT_WAVE_TOTAL_LENGTH"]()

In [ ]:
ds_daily = xarray.Dataset()
_resample = ds_hourly["tas"].resample(time="1D")
ds_daily["tasmin"] = _resample.min()
ds_daily["tasmax"] = _resample.max()

## How does the comparison of xarray DataArray objects with different dimensions work?

At first I explored the source code of the indicator to understand how comparisons and countings are performed.
All narrows down to the computation of [the heat_wave_total_length index](https://xclim.readthedocs.io/en/stable/indices.html#xclim.indices.heat_wave_total_length) which performs a comparison between data and parameters. Since here parameters are xarray DataArray objects, that operation is broadcasted (read: parallelised over all elements) along their dimensions. This suggested me that when xarray DataArray objects are used as arguments, they need to have the same shape.

This question came to me because I wanted to understand in practice how the broadcasting of the comparison between xarray DataArray objects is performed.

In [ ]:
a = ds_daily["tasmin"]
b = xarray.DataArray([33.4, -234.6, 50.9], attrs={"units": "degC", "standard_name": "air_temperature"})
c = a >= b

display(a)
display(c)

## Evaluate indicator by passing xarray DataArray objects

Now I test the evaluation of the indicator.
First I create mock up data to use as values of parameters. The parameters which can be supplied as xarray DataArray objects are `thresh_tasmin` and `thresh_tasmin`. In this example the latter it a subset of the former, for ease.

In [ ]:
m = numpy.meshgrid([.1, .3, .4, .6, .8], [1, 2, 3, 4.3])
display(m)

t = xarray.DataArray(
    data=m,
    attrs={"units": "degC"}
)
display(t)

I can confirm that the only constraint on the xarray DataArray objects is to have the same shape. Here it is shown by using different slicings with same shape of the same xarrayDataArray object as arguments.

From this observation I suspect that the evaluation is pointwise: a single value is associated to each parameter, in particular the values corresponding to the same position in the xarray DataArray objects (i.e. with same coordinates).

In [ ]:
t1 = t[:, 1:3, 2:4]
t2 = t[:, 2:4, 2:4]

res = indicator(
    ds=ds_daily,
    thresh_tasmin=t1,
    thresh_tasmax=t2
).mean(dim="time", keep_attrs=True)
display(res)

As explained in the changelog, when xarray DataArray objects are used as arguments, the description of the xarray DataArray object resulting from the application of the indicator does not contain the single values for those parameters, a generic string is shown instead.

In [ ]:
print(res.attrs["description"])

Now I show whether the evaluation of the indicator is pointwise on the multidimensional arguments.

Two reference values are selected from the xarray DataArray objects assigned as parameters. Since the two xarray DataArray objects are pointwise different by construction, I just set a single reference position to get two different values.

I use those values to compare the correctness of the different ways to evaluate the indicator:

- I evaluate the indicator by passing the reference values singularly.
- I evaluate the indicator by passing the xarray DataArray objects as arguments, then I select only the element of the resulting object corresponding to the reference position.

This is knowledge to leverage: it means that the evaluation of indicators can be parallelised on parameters and the key to do that is in the source code of any indicator which allows xarray DataArray objects as arguments for parameters.

In [ ]:
reference_position = (1, 1, 1)
print("Reference arguments are different by construction.")
display(t1[reference_position])
display(t2[reference_position])

print("Outcome of evaluation with single arguments.")
a = indicator(
    ds=ds_daily,
    thresh_tasmin=t1[reference_position],
    thresh_tasmax=t2[reference_position]
).mean(dim="time", keep_attrs=True)
display(a)

print("Outcome of evaluation with multidimensional arguments.")
c = indicator(
    ds=ds_daily,
    thresh_tasmin=t1,
    thresh_tasmax=t2
).mean(dim="time", keep_attrs=True)
b = c.isel({f"dim_{i}": item for i, item in enumerate(reference_position)})
display(b)

print("Are outcomes equal?", a.equals(b))